In [9]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langchain_ollama import ChatOllama
from pydantic import BaseModel , Field
import operator

In [10]:
model = ChatOllama(model = "llama3.2")

In [11]:
class EvaluationSchema(BaseModel):
  feedback: str = Field(description = "Detailed feedback for the essay, highlighting strengths and areas for improvement.")
  score:int = Field(description = "Score out of 10" , ge = 0 , le = 10)

In [12]:
structured_model = model.with_structured_output(EvaluationSchema)

In [13]:
essay = """The modern world operates on a seductive premise: more choices lead to more freedom, and more freedom leads to greater happiness. From the thousands of movies streaming on our screens to the endless variations of a simple cup of coffee, choice defines contemporary life. However, this abundance often produces the exact opposite effect. As psychologist Barry Schwartz famously noted, an overabundance of options can paralyze consumers and diminish their ultimate satisfaction. In an era of unlimited possibilities, the human mind increasingly struggles with the exhaustion of decision-making, revealing that choice is a paradox.First, an overwhelming number of options leads to decision paralysis rather than freedom. When faced with a few distinct alternatives, comparing features and making a selection is relatively simple. When those options multiply into the hundreds, the cognitive load required to evaluate them becomes overwhelming. Consumers find themselves frozen, spending hours scrolling through streaming platforms or reading endless product reviews without ever making a choice. The fear of making a suboptimal decision creates anxiety, turning what should be a liberating experience into a stressful chore.Second, even when a choice is finally made, large selection pools decrease overall satisfaction. When options are plentiful, it is easy to imagine that an unchosen alternative might have been better. This phenomenon, known as opportunity cost, breeds immediate regret. If a consumer buys a smartphone out of two available options, they are likely content. If they buy one out of fifty options, they constantly wonder if one of the other forty-nine models possessed superior battery life or a better camera. The presence of alternatives makes it easy to find fault with the chosen path, eroding the joy of ownership.Finally, an excess of choice forces individuals to escalate their expectations layout. When there are countless variations of a product or service, consumers assume that the perfect option must exist. If the chosen item is anything less than flawless, they blame themselves for making a poor selection. In a world with limited options, a mediocre outcome is viewed as bad luck; in a world of infinite options, a mediocre outcome is viewed as a personal failure. This shift in accountability damages mental well-being, transforming abundance into a source of self-doubt.Ultimately, while the freedom to choose is a fundamental human right, unlimited choice is a psychological burden. The digital age has conquered scarcity, but it has replaced it with the tyranny of excess. To find peace in a world of endless options, individuals must learn to limit their parameters and practice gratitude for "good enough" rather than chasing the illusion of perfection. True freedom in the modern world is not the ability to choose everything, but the wisdom to intentionally choose less."""

In [14]:
class UPSCState(TypedDict):
    essay:str
    language_feedback:str
    cot_feedback:str
    doa_feedback:str
    final_feedback:str
    
    individual_score: Annotated[list[int] , operator.add]
    final_avg_score: float

In [15]:
def evaluate_language(state:UPSCState):
  prompt = f'Evaluate the language quality of the following essay, for this provide a proper feedback and a score out of 10. The essay is: {state["essay"]}'
  
  structured_output = structured_model.invoke(prompt)
  
  return {'language_feedback':structured_output.feedback , 'individual_score':[structured_output.score]}

In [16]:
def evaluate_analysis(state: UPSCState):
  prompt = f'Evaluate the analysis quality of the following essay, for this provide a proper feedback and a score out of 10. The essay is: {state["essay"]}'
  
  structured_output = structured_model.invoke(prompt)
  
  return{'doa_feedback':structured_output.feedback , 'individual_score':[structured_output.score]}

In [17]:
def evaluate_thought(state:UPSCState):
  
  prompt = f'Evaluate the thought quality of the following essay, for this provide a proper feedback and a score out of 10. The essay is: {state["essay"]}'
  
  structured_output = structured_model.invoke(prompt)
  
  return {'cot_feedback':structured_output.feedback , 'individual_score':[structured_output.score]}

In [18]:
def final_evaluation(state:UPSCState):
  
  prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["doa_feedback"]} \n clarity of thought feedback - {state["cot_feedback"]}'
  overall_feedback = model.invoke(prompt)
  
  avg_score = sum(state["individual_score"])/len(state["individual_score"])
  
  return {'final_feedback':overall_feedback , 'final_avg_score':avg_score}

In [19]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language' , evaluate_language)
graph.add_node('evaluate_analysis' , evaluate_analysis)
graph.add_node('evaluate_thought' , evaluate_thought)
graph.add_node('final_evaluation' , final_evaluation)


graph.add_edge(START , 'evaluate_language')
graph.add_edge(START , 'evaluate_analysis')
graph.add_edge(START , 'evaluate_thought')

graph.add_edge('evaluate_language' , 'final_evaluation')
graph.add_edge('evaluate_analysis' , 'final_evaluation')
graph.add_edge('evaluate_thought' , 'final_evaluation')


graph.add_edge('final_evaluation' , END)

workflow = graph.compile()

In [23]:
initial_state ={
  'essay' : essay
}

final_state = workflow.invoke(initial_state)

In [24]:
print(final_state)

{'essay': 'The modern world operates on a seductive premise: more choices lead to more freedom, and more freedom leads to greater happiness. From the thousands of movies streaming on our screens to the endless variations of a simple cup of coffee, choice defines contemporary life. However, this abundance often produces the exact opposite effect. As psychologist Barry Schwartz famously noted, an overabundance of options can paralyze consumers and diminish their ultimate satisfaction. In an era of unlimited possibilities, the human mind increasingly struggles with the exhaustion of decision-making, revealing that choice is a paradox.First, an overwhelming number of options leads to decision paralysis rather than freedom. When faced with a few distinct alternatives, comparing features and making a selection is relatively simple. When those options multiply into the hundreds, the cognitive load required to evaluate them becomes overwhelming. Consumers find themselves frozen, spending hours